# Grafo de conocimiento integrando distintas fuentes

## Fuentes de Integración

Integra datos de Wikidata con ontologías musicales externas:
- Wikidata: Datos estructurados vía SPARQL
- MusicBrainz REST API: Discografía detallada (JSON→RDF)
- Music Ontology: Vocabulario formal
- Schema.org: Propiedades estándar



Importar Librerías

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
from rdflib.namespace import FOAF, DCTERMS
import requests
import json
import time
from datetime import datetime
import sbc_tools as sbc

## Inicialización del grafo de conocimiento

Crear grafo principal y definir todos los namespaces necesarios

In [ ]:
graph = Graph()

WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
MO = Namespace("http://purl.org/ontology/mo/")
SCHEMA = Namespace("https://schema.org/")
MB = Namespace("https://musicbrainz.org/")
LOCAL = Namespace("http://informatica.ucm.es/musicontology/")

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(SPARQL_JSON)

graph.bind("wd", WD)
graph.bind("wdt", WDT)
graph.bind("mo", MO)
graph.bind("schema", SCHEMA)
graph.bind("mb", MB)
graph.bind("local", LOCAL)
graph.bind("foaf", FOAF)
graph.bind("dcterms", DCTERMS)
graph.bind("owl", OWL)
graph.bind("rdfs", RDFS)

---

# Crear Jerarquía de Clases Unificada

## Crea jerarquía de clases unificando diferentes ontologías

### Declarar ontología base

In [ ]:
graph.add((LOCAL[""], RDF.type, OWL.Ontology))
graph.add((LOCAL[""], RDFS.label, Literal("Ontología Musical Unificada")))
graph.add((LOCAL[""], RDFS.comment, Literal("Integración de Wikidata, Music Ontology y Schema.org")))

## CLASES BASE

### Artista Musical (unifica conceptos de diferentes ontologías)

In [ ]:
graph.add((LOCAL.MusicalArtist, RDF.type, OWL.Class))
graph.add((LOCAL.MusicalArtist, RDFS.label, Literal("Artista Musical")))

### Mapear a ontologías externas (owl:equivalentClass)

In [ ]:
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, MO.MusicArtist))
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, SCHEMA.MusicGroup))
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, MO.MusicArtist))
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, SCHEMA.MusicGroup))


### Subclases de Artista Musical (ejemplo)

In [ ]:
artist_types = [
    ("SoloArtist", "Artista en solitario", MO.SoloMusicArtist),
    ("Band", "Banda musical", MO.MusicGroup)
]
        
for class_name, label, external_class in artist_types:
    class_uri = LOCAL[class_name]
    graph.add((class_uri, RDF.type, OWL.Class))
    graph.add((class_uri, RDFS.subClassOf, LOCAL.MusicalArtist))
    graph.add((class_uri, RDFS.label, Literal(label)))
    if external_class:
        graph.add((class_uri, OWL.equivalentClass, external_class))

## OBRAS MUSICALES

In [ ]:
graph.add((LOCAL.MusicalWork, RDF.type, OWL.Class))
graph.add((LOCAL.MusicalWork, RDFS.label, Literal("Obra Musical")))
graph.add((LOCAL.MusicalWork, OWL.equivalentClass, MO.MusicalWork))
graph.add((LOCAL.MusicalWork, OWL.equivalentClass, SCHEMA.MusicComposition))

### Subclases de Obras Musicales (ejemplo)

In [ ]:
work_types = [
    ("Album", "Álbum", MO.Record, SCHEMA.MusicAlbum),
    ("Song", "Canción", MO.Track, SCHEMA.MusicRecording)
]

for class_name, label, mo_class, schema_class in work_types:
    class_uri = LOCAL[class_name]
    graph.add((class_uri, RDF.type, OWL.Class))
    graph.add((class_uri, RDFS.subClassOf, LOCAL.MusicalWork))
    graph.add((class_uri, RDFS.label, Literal(label)))
    if mo_class:
        graph.add((class_uri, OWL.equivalentClass, mo_class))
    if schema_class:
        graph.add((class_uri, OWL.equivalentClass, schema_class))

## GÉNEROS MUSICALES

In [ ]:
graph.add((LOCAL.MusicGenre, RDF.type, OWL.Class))
graph.add((LOCAL.MusicGenre, RDFS.label, Literal("Género Musical")))
graph.add((LOCAL.MusicGenre, OWL.equivalentClass, MO.Genre))
graph.add((LOCAL.MusicGenre, OWL.equivalentClass, SCHEMA.MusicGenre))

Usaremos el código de la práctica anterior para visualizar la ontología creada

In [ ]:
sbc.show_graph(graph, output_file="grafo_ontologia_musical_jerarquia.html")

---

# Crear Propiedades Unificadas

## Define propiedades unificadas mapeando a ontologías externas

## PROPIEDADES DE OBJETO

### Propiedad: creador/artista

In [ ]:
graph.add((LOCAL.creator, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.creator, RDFS.label, Literal("creador")))
graph.add((LOCAL.creator, OWL.equivalentProperty, MO.composer))
graph.add((LOCAL.creator, OWL.equivalentProperty, SCHEMA.byArtist))
graph.add((LOCAL.creator, OWL.equivalentProperty, DCTERMS.creator))

### Propiedad: género musical

In [ ]:
graph.add((LOCAL.hasGenre, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.hasGenre, RDFS.label, Literal("tiene género")))
graph.add((LOCAL.hasGenre, RDFS.domain, LOCAL.MusicalWork))
graph.add((LOCAL.hasGenre, RDFS.range, LOCAL.MusicGenre))
graph.add((LOCAL.hasGenre, OWL.equivalentProperty, MO.genre))
graph.add((LOCAL.hasGenre, OWL.equivalentProperty, SCHEMA.genre))

### Propiedad: miembro de banda

In [ ]:
graph.add((LOCAL.memberOf, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.memberOf, RDFS.label, Literal("miembro de")))
graph.add((LOCAL.memberOf, RDFS.domain, LOCAL.SoloArtist))
graph.add((LOCAL.memberOf, RDFS.range, LOCAL.Band))
graph.add((LOCAL.memberOf, OWL.equivalentProperty, MO.member_of))
graph.add((LOCAL.memberOf, OWL.equivalentProperty, SCHEMA.memberOf))

## PROPIEDADES DE DATOS

### Fecha de publicación

In [ ]:
graph.add((LOCAL.releaseDate, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.releaseDate, RDFS.label, Literal("fecha de lanzamiento")))
graph.add((LOCAL.releaseDate, RDFS.domain, LOCAL.MusicalWork))
graph.add((LOCAL.releaseDate, RDFS.range, XSD.date))
graph.add((LOCAL.releaseDate, OWL.equivalentProperty, SCHEMA.datePublished))

### Duración

In [ ]:
graph.add((LOCAL.duration, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.duration, RDFS.label, Literal("duración")))
graph.add((LOCAL.duration, RDFS.domain, LOCAL.Song))
graph.add((LOCAL.duration, RDFS.range, XSD.duration))
graph.add((LOCAL.duration, OWL.equivalentProperty, MO.duration))
graph.add((LOCAL.duration, OWL.equivalentProperty, SCHEMA.duration))

### MusicBraind ID

In [ ]:
graph.add((LOCAL.mbid, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.mbid, RDFS.label, Literal("MusicBrainz ID")))

### Popularidad

In [ ]:
graph.add((LOCAL.popularity, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.popularity, RDFS.label, Literal("popularidad")))
graph.add((LOCAL.popularity, RDFS.range, XSD.integer))

---

# Obtener artistas desde Wikidata

## Obtiene artistas musicales de Wikidata

**Parámetros:**
- `genre_name`: Género musical a buscar
- `limit`: Número máximo de resultados

**Consulta SPARQL:**
- Busca humanos (Q5) que sean músicos (Q639669)
- Filtra por género musical especificado
- Obtiene información adicional: país, fecha de nacimiento


In [ ]:
def get_artists_from_wikidata(genre_name="pop music", limit=10):

    query = f"""
        SELECT DISTINCT ?artist ?artistLabel ?genreLabel ?countryLabel ?birthDate ?mbid WHERE {{
        ?artist wdt:P31 wd:Q5 ;
                wdt:P106 wd:Q639669 ;
                wdt:P136 ?genre .

        ?artist rdfs:label ?artistLabel .
        ?genre rdfs:label ?genreLabel .
        FILTER (lang(?genreLabel) = 'en')
        FILTER(CONTAINS(LCASE(?genreLabel), "{genre_name.lower()}"))

        OPTIONAL {{ ?artist wdt:P27 ?country . 
                    ?country rdfs:label ?countryLabel
                    FILTER (lang(?countryLabel) = 'es')
                }}
        OPTIONAL {{ ?artist wdt:P569 ?birthDate . }}
        OPTIONAL {{ ?artist wdt:P434 ?mbid . }}
        FILTER (lang(?artistLabel) = 'es')
        

        }}
        LIMIT {limit}
        """

    sparql.setQuery(query)
    results = sparql.query().convert()

    artists_data = []
    for result in results["results"]["bindings"]:
        artists_data.append({
            'uri': result["artist"]["value"],
            'name': result["artistLabel"]["value"],
            'genre': result.get("genreLabel", {}).get("value", "Desconocido"),
            'country': result.get("countryLabel", {}).get("value", "Desconocido"),
            'birthDate': result.get("birthDate", {}).get("value", None),
            'mbid': result.get("mbid", {}).get("value", None)
        })
    
    return artists_data

In [ ]:
# lo probamos
artists = get_artists_from_wikidata(genre_name="pop music", limit=5)
for artist in artists:
    print(artist)

---

# Integrar Artistas  de Wikidata

## Integra datos de artistas de Wikidata en la ontología unificada

**Para cada artista:**
1. Añade como instancia de la clase unificada
2. Mapea a clases de ontologías externas
3. Añade propiedades usando vocabularios estándar (FOAF, Schema.org)
4. Incluye información contextual (país, fecha de nacimiento, género)

In [ ]:
def integrate_wikidata_artists(artists_data):
    
    for artist in artists_data:
        artist_uri = URIRef(artist['uri'])

        graph.add((artist_uri, RDF.type, LOCAL.SoloArtist))
        graph.add((artist_uri, RDFS.label, Literal(artist['name'], lang="es")))

        graph.add((artist_uri, RDF.type, MO.MusicArtist))
        graph.add((artist_uri, RDF.type, SCHEMA.MusicGroup))

        graph.add((artist_uri, FOAF.name, Literal(artist['name'])))
        
        if artist['country'] != "Desconocido":
            graph.add((artist_uri, LOCAL.country, Literal(artist['country'])))
            graph.add((artist_uri, SCHEMA.nationality, Literal(artist['country'])))

        if artist['birthDate']:
            birth_date = artist['birthDate'][:10]
            graph.add((artist_uri, SCHEMA.birthDate,
                        Literal(birth_date, datatype=XSD.date)))
        
        if artist['genre']:
            genre_uri = LOCAL[artist['genre'].replace(" ", "_")]
            graph.add((genre_uri, RDF.type, LOCAL.MusicGenre))
            graph.add((genre_uri, RDFS.label, Literal(artist['genre'])))
            graph.add((artist_uri, LOCAL.hasGenre, genre_uri))


In [ ]:
integrate_wikidata_artists(artists)

In [ ]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_base.html")

---

# Enriquecer con MusicBrainz REST API

URL base

In [ ]:
mb_base_url = "https://musicbrainz.org/ws/2/"
mb_headers = {
    'User-Agent': 'MusicOntologyIntegrator/1.0 (educational purpose)',
    'Accept': 'application/json'
}

## Buscar artista en MusicBrainz por nombre

Endpoint: https://musicbrainz.org/ws/2/artist/

In [ ]:
def search_artist_musicbrainz(artist_name):
        
        url = f"{mb_base_url}artist/"
        params = {
            'query': artist_name,
            'fmt': 'json',
            'limit': 1
        }
        
        try:
            response = requests.get(url, params=params, headers=mb_headers)
            time.sleep(1)
            
            if response.status_code == 200:
                data = response.json()
                if data.get('artists') and len(data['artists']) > 0:
                    artist = data['artists'][0]
                    return {
                        'mbid': artist.get('id'),
                        'name': artist.get('name'),
                        'type': artist.get('type'),
                        'country': artist.get('country'),
                        'life_span': artist.get('life-span', {})
                    }
        except Exception as e:
            print(f"Error: {e}")
        
        return None

In [ ]:
# lo probamos
search_artist_musicbrainz("Raphael")

## Obtener releases (álbumes) del artista

Endpoint: https://musicbrainz.org/ws/2/release/

In [ ]:
def get_releases_musicbrainz(mbid):
        
    if not mbid:
        return None
    
    url = f"{mb_base_url}release/"
    params = {
        'artist': mbid,
        'type': 'album',
        'fmt': 'json',
        'limit': 10
    }
    
    try:
        response = requests.get(url, params=params, headers=mb_headers)
        time.sleep(1)
        
        if response.status_code == 200:
            data = response.json()
            releases = []
            
            for release in data.get('releases', []):
                releases.append({
                    'mbid': release.get('id'),
                    'title': release.get('title'),
                    'date': release.get('date'),
                    'country': release.get('country'),
                    'status': release.get('status')
                })
            
            return releases
    except Exception as e:
        print(f"Error: {e}")
    
    return None

In [ ]:
# lo probamos
get_releases_musicbrainz('b34cfe02-994d-40dc-854f-bf54a23dd405')  # MBID de Raphael

## Obtener recordings (canciones) de un release

Endpoint: https://musicbrainz.org/ws/2/release/{mbid}?inc=recordings

In [ ]:
def get_recordings_from_release(release_mbid):
        
    if not release_mbid:
        return None

    url = f"{mb_base_url}release/{release_mbid}"
    params = {
        'inc': 'recordings',
        'fmt': 'json'
    }
    
    try:
        response = requests.get(url, params=params, headers=mb_headers)
        time.sleep(1)
        
        if response.status_code == 200:
            data = response.json()
            recordings = []
            
            for medium in data.get('media', []):
                for track in medium.get('tracks', []):
                    recording = track.get('recording', {})
                    recordings.append({
                        'mbid': recording.get('id'),
                        'title': recording.get('title'),
                        'length': recording.get('length')
                    })
            
            return recordings
    except Exception as e:
        print(f"Error: {e}")
    
    return None

In [ ]:
# lo probamos
get_recordings_from_release('2a3dac67-22ad-42d3-b459-a6a69cf56d2b')

---

# Integrar datos MusicBrainz al grafo

Podemos utilizar dos opciones:
- Utilizar los datos (mbid, nombre, ...) obtenidos de wikidata
- Integrar directamente por nombre de autor, buscando en el api de MusicBrainz con el método ``search_artist_musicbrainz()``

El flujo es el siguiente:
Integrar información Artista >> Integrar información albumes >> Integrar información canciones

In [ ]:
def add_musicbrainz_info_to_graph(artist_uri, artist_name, wikidata_mbid=None):
    if wikidata_mbid:
        mb_data = {
            'mbid': wikidata_mbid,
            'name': artist_name
        }
    else:
        mb_data = search_artist_musicbrainz(artist_name)
    
    if not mb_data or not mb_data.get('mbid'):
        return False
    
    artist_uri_ref = URIRef(artist_uri)
    mbid = mb_data['mbid']
    
    graph.add((artist_uri_ref, LOCAL.mbid, Literal(mbid)))
    
    mb_artist_uri = URIRef(f"https://musicbrainz.org/artist/{mbid}")
    graph.add((artist_uri_ref, OWL.sameAs, mb_artist_uri))
    
    if mb_data.get('type'):
        graph.add((artist_uri_ref, LOCAL.artistType, Literal(mb_data['type'])))
    
    if mb_data.get('country'):
        graph.add((artist_uri_ref, SCHEMA.nationality, Literal(mb_data['country'])))
    
    add_musicbrainz_albums_to_graph(mbid, artist_uri_ref, limit=5)


def add_musicbrainz_albums_to_graph(artist_mbid, artist_uri_ref, limit = 5):
    releases = get_releases_musicbrainz(artist_mbid)
    
    if not releases:
        return False

    for release in releases[:limit]:
        release_uri = URIRef(f"https://musicbrainz.org/release/{release['mbid']}")

        graph.add((release_uri, RDF.type, LOCAL.Album))
        graph.add((release_uri, RDF.type, MO.Record))
        graph.add((release_uri, DCTERMS.title, Literal(release['title'])))
        graph.add((release_uri, LOCAL.creator, artist_uri_ref))
        graph.add((release_uri, MO.maker, artist_uri_ref))
        graph.add((release_uri, LOCAL.mbid, Literal(release['mbid'])))
        
        if release.get('date'):
            try:
                date_str = release['date']
                if len(date_str) >= 10:
                    graph.add((release_uri, LOCAL.releaseDate, Literal(date_str[:10], datatype=XSD.date)))
            except:
                pass
        
        if release.get('country'):
            graph.add((release_uri, LOCAL.releaseCountry, Literal(release['country'])))
        
        add_musicbrainz_recordings_to_graph(release['mbid'], artist_uri_ref, limit=3)

        return True


def add_musicbrainz_recordings_to_graph(release_mbid, artist_uri_ref, limit = 3):
        recordings = get_recordings_from_release(release_mbid)
        if recordings:
            for recording in recordings[:limit]:
                recording_uri = URIRef(f"https://musicbrainz.org/recording/{recording['mbid']}")
                
                graph.add((recording_uri, RDF.type, LOCAL.Song))
                graph.add((recording_uri, RDF.type, MO.Track))
                graph.add((recording_uri, DCTERMS.title, Literal(recording['title'])))
                graph.add((recording_uri, LOCAL.creator, artist_uri_ref))
                
                if recording.get('length'):
                    duration_ms = int(recording['length'])
                    duration_seconds = duration_ms / 1000
                    graph.add((recording_uri, LOCAL.durationSeconds,
                                    Literal(duration_seconds, datatype=XSD.decimal)))
        
        return True

In [ ]:
# Lo probamos con un solo artista

#primero añadimos sus datos de Wikidata
integrate_wikidata_artists([{
    'uri': 'http://www.wikidata.org/entity/Q180686',
    'name': 'Raphael',
    'genre': 'pop music',
    'country': 'Spain',
    'birthDate': '1943-05-05',
    'mbid': 'b34cfe02-994d-40dc-854f-bf54a23dd405'}])

add_musicbrainz_info_to_graph('http://www.wikidata.org/entity/Q180686', 'Raphael', 'b34cfe02-994d-40dc-854f-bf54a23dd405')

In [ ]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_con_raphael.html")

In [ ]:
# Lo probamos con todos los artistas obtenidos de Wikidata
for artist in artists:
    add_musicbrainz_info_to_graph(artist['uri'], artist['name'], artist.get('mbid'))
    time.sleep(1)  # Para no saturar el API

In [ ]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_con_artistas_wikidata.html")

---

# Cargar Ontología Externa

## Carga una ontología externa con vocabulario adicional e integrarla.

### Utilizamos como ejemplo la ontología Music Ontology: http://musicontology.com/specification/index.html

**Proceso:**
1. Crea un grafo temporal para la ontología externa
2. Parsea el archivo/URL en el formato especificado
3. Fusiona los triples con el grafo principal

In [ ]:
ontology_url = "http://purl.org/ontology/mo/"
external_graph = Graph()
external_graph.parse(ontology_url, format='xml')
for triple in external_graph:
    graph.add(triple)


## Añadimos los mapeos

In [ ]:
graph.add((LOCAL[""], OWL.imports, URIRef("http://purl.org/ontology/mo/")))

mappings = [
    (LOCAL.MusicalArtist, MO.MusicArtist, "Artista Musical"),
    (LOCAL.Album, MO.Record, "Álbum"),
    (LOCAL.Song, MO.Track, "Canción"),
]

for local_class, mo_class, label in mappings:
    graph.add((local_class, OWL.equivalentClass, mo_class))
    graph.add((local_class, RDFS.label, Literal(label, lang="es")))

In [ ]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_con_mo.html")

### Guardar la ontología

In [ ]:
sbc.save(graph, "grafo_ontologia_musical.ttl", format='turtle')

---

# Consultas sobre la Ontología Integrada

## Ejemplo de consulta SPARQL sobre la ontología integrada

**Objetivo:** Encontrar todos los artistas y sus álbumes en la ontología

In [ ]:
query1 = f"""
PREFIX local: <{LOCAL}>
        PREFIX rdfs: <{RDFS}>
        PREFIX dcterms: <{DCTERMS}>

SELECT ?artist ?artistName ?album ?albumTitle WHERE {{
    ?artist a local:SoloArtist ;
            rdfs:label ?artistName .
     ?album local:creator ?artist ;
            dcterms:title ?albumTitle .
}}
"""

print("Artistas con sus álbumes:")
results = graph.query(query1)
for row in results:
    artist = row.artistName
    album = row.albumTitle
    print(f"- {artist} - Álbum: {album}")

    

# Ejercicio

## Crea un grafo de conocimiento para el dominio que consideres más oportuno integrando conocimiento de varias fuentes de conocimiento semántico.

# Alumnos
## Pedro Antonio Martín Peláez
## Álvaro Ray Martínez 

In [1]:
from rdflib import Graph, URIRef, Namespace, RDF, RDFS, OWL
import sbc_tools as sbc

In [2]:
#carga de grafos
g_main = Graph()
g_main.parse("data/anime_ontology_ext.ttl", format="turtle")

g_ext = Graph()
g_ext.parse("data/anime-ontology-external.ttl", format="turtle")

print("Grafo principal:", len(g_main), "triples")
print("Ontología externa:", len(g_ext), "triples")

#miramos las clases de mabos grafos
def get_classes(graph):
    return set(graph.subjects(RDF.type, OWL.Class))

main_classes = get_classes(g_main)
ext_classes = get_classes(g_ext)

print("Clases en mi grafo:", len(main_classes))
print("Clases externas:", len(ext_classes))

#mapeamos clases equivalentes label = label
def label(graph, cls):
    for lbl in graph.objects(cls, RDFS.label):
        return str(lbl).lower()
    return None

mapping = {}

for c_main in main_classes:
    lbl_main = label(g_main, c_main)
    if not lbl_main:
        continue

    for c_ext in ext_classes:
        lbl_ext = label(g_ext, c_ext)
        if lbl_ext and lbl_main == lbl_ext:
            mapping[c_main] = c_ext
            break

print("Clases mapeadas:", len(mapping))

#mapeo de las propiedades equivalentes 
def print_class_properties(graph, cls):
    print(f"\nPropiedades para {cls}:")
    props = set()
    for s, p, o in graph.triples((None, RDF.type, cls)):
        for prop, val in graph.predicate_objects(s):
            props.add(prop)
    for p in props:
        print(" -", p)
AnimeSeries = URIRef("http://www.fdi.ucm.es/mia/sbc/animeontology/AnimeSeries")
Programme = URIRef("http://purl.org/ontology/po/Programme")

print_class_properties(g_main, AnimeSeries)
print_class_properties(g_ext, Programme)

ONTO_hasGenre = URIRef("http://www.fdi.ucm.es/mia/sbc/animeontology/hasGenre")
PO_hasGenre = URIRef("http://www.semanticweb.org/eladi/ontologies/anime-ontology#hasGenre")
g_main.add((ONTO_hasGenre, OWL.equivalentProperty, PO_hasGenre))

#añadimos las clases que no son equivalentes
for c_main, c_ext in mapping.items():
    triple = (c_main, OWL.equivalentClass, c_ext)
    if triple not in g_main:
        g_main.add(triple)


new_classes = ext_classes - main_classes

for cls in new_classes:
    for p, o in g_ext.predicate_objects(cls):
        triple = (cls, p, o)
        if triple not in g_main:
            g_main.add(triple)

print("Clases añadidas:", len(new_classes))

#traemos las propiedades
ext_properties = set(g_ext.subjects(RDF.type, OWL.ObjectProperty)) | \
                 set(g_ext.subjects(RDF.type, OWL.DatatypeProperty))

main_properties = set(g_main.subjects(RDF.type, OWL.ObjectProperty)) | \
                  set(g_main.subjects(RDF.type, OWL.DatatypeProperty))

new_props = ext_properties - main_properties

for prop in new_props:
    for p, o in g_ext.predicate_objects(prop):
        triple = (prop, p, o)
        if triple not in g_main:
            g_main.add(triple)

print("Propiedades añadidas:", len(new_props))

#lo mismo con los individuos
ext_individuals = set(g_ext.subjects(RDF.type, OWL.NamedIndividual))
main_individuals = set(g_main.subjects(RDF.type, OWL.NamedIndividual))

new_individuals = ext_individuals - main_individuals

for ind in new_individuals:
    for p, o in g_ext.predicate_objects(ind):
        triple = (ind, p, o)
        if triple not in g_main:
            g_main.add(triple)

print("Individuos añadidos:", len(new_individuals))


sbc.save(g_main, "grafo_fusionado.ttl", format='turtle')

Grafo principal: 19423 triples
Ontología externa: 1356 triples
Clases en mi grafo: 2
Clases externas: 35
Clases mapeadas: 2

Propiedades para http://www.fdi.ucm.es/mia/sbc/animeontology/AnimeSeries:
 - http://www.fdi.ucm.es/mia/sbc/animeontology/numSeasons
 - http://www.fdi.ucm.es/mia/sbc/animeontology/numEpisodes
 - http://www.w3.org/1999/02/22-rdf-syntax-ns#type
 - http://www.w3.org/2000/01/rdf-schema#label
 - http://www.fdi.ucm.es/mia/sbc/animeontology/hasGenre

Propiedades para http://purl.org/ontology/po/Programme:
 - http://www.semanticweb.org/eladi/ontologies/anime-ontology#producedBy
 - http://www.semanticweb.org/eladi/ontologies/anime-ontology#firstAiring
 - http://www.semanticweb.org/eladi/ontologies/anime-ontology#hasEnding
 - http://www.semanticweb.org/eladi/ontologies/anime-ontology#runtime
 - http://www.ontologydesignpatterns.org/cp/owl/bag.owl#size
 - http://www.semanticweb.org/eladi/ontologies/anime-ontology#hasGenre
 - http://www.semanticweb.org/eladi/ontologies/anime-

In [3]:
sbc.print_ontology_stats(g_main)


=== Estadísticas de la Ontología ===
Total de triples: 20216
Clases OWL: 37
Propiedades de objeto: 24
Instancias: 4444
Relaciones de subclase: 26


In [4]:
g = g_main
sbc.print_class_hierarchy(g)


sbc.show_graph(g, output_file="anime_ontology_external.html")


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Category (Category)
  📁 genre (Genre)
📁 Sin etiqueta (ned05e8b416524fc5b3c5eacd59f30a05b1)
📁 Genre (Genre)
  └─ ASMR: ASMR
  └─ Monster_of_the_Week: Monster of the Week
  └─ Sentai: Sentai
  └─ Westerns_on_television: Westerns on television
  └─ Xianxia: Xianxia
  └─ action_comics: action comics
  └─ action_manga: action manga
  └─ adventure_manga: adventure manga
  └─ adventure_video_game: adventure video game
  └─ alternate_history: alternate history
  └─ alternate_history_television_series: alternate history television series
  └─ animal-themed_television_series: animal-themed television series
  └─ animated_cartoon: animated cartoon
  └─ anime-influenced_animation: anime-influenced animation
  └─ manga_style: anime/manga style
  └─ anthology_series: anthology series
  └─ apocalyptic_fiction: apocalyptic fiction
  └─ biographical_film: biographical film
  └─ bishōjo_game: bishōjo game
  └─ chuanyue: chuanyue
  └─ comedy_drama_anime: comedy dram

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, XSD, URIRef
import requests
import re
import time

def clean_uri(name):
    name = name.strip()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[^a-zA-Z0-9_]', '', name)
    return name

# Cargar grafo existente
g = sbc.load("grafo_fusionado.ttl", format="turtle")

# Namespaces
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
NS2 = Namespace("http://www.semanticweb.org/eladi/ontologies/anime-ontology#")

for anime_id in range(1, 101):
    try:
        anime_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}").json()['data']
    except:
        print(f"Anime ID {anime_id} error")
        continue

    title = anime_data.get('title_english') or anime_data['title']
    anime_uri = ONTO[clean_uri(title)]

    # AnimeSeries
    if (anime_uri, RDF.type, ONTO.AnimeSeries) not in g:
        g.add((anime_uri, RDF.type, ONTO.AnimeSeries))

    # Títulos
    for key, value in [('englishTitle', anime_data.get('title_english')), ('episodeTitle', anime_data['title'])]:
        if value:
            pred = getattr(NS2, key)
            lit = Literal(value, datatype=XSD.string)
            if (anime_uri, pred, lit) not in g:
                g.add((anime_uri, pred, lit))

    # Número de episodios y fecha
    if anime_data.get('episodes'):
        lit = Literal(anime_data['episodes'], datatype=XSD.integer)
        if (anime_uri, ONTO.numEpisodes, lit) not in g:
            g.add((anime_uri, ONTO.numEpisodes, lit))
    if anime_data.get('aired') and anime_data['aired'].get('from'):
        lit = Literal(anime_data['aired']['from'], datatype=XSD.dateTime)
        if (anime_uri, ONTO.startDate, lit) not in g:
            g.add((anime_uri, ONTO.startDate, lit))

    # Géneros
    for genre in anime_data.get('genres', []):
        genre_uri = ONTO[clean_uri(genre['name'])]
        if (genre_uri, RDF.type, ONTO.Genre) not in g:
            g.add((genre_uri, RDF.type, ONTO.Genre))
            g.add((genre_uri, NS2.label, Literal(genre['name'], datatype=XSD.string)))
        if (anime_uri, ONTO.hasGenre, genre_uri) not in g:
            g.add((anime_uri, ONTO.hasGenre, genre_uri))

    # Estudios
    for studio in anime_data.get('studios', []):
        studio_uri = ONTO[clean_uri(studio['name'])]
        if (studio_uri, RDF.type, NS2.Studio) not in g:
            g.add((studio_uri, RDF.type, NS2.Studio))
            g.add((studio_uri, NS2.label, Literal(studio['name'], datatype=XSD.string)))
        if (studio_uri, NS2.produced, anime_uri) not in g:
            g.add((studio_uri, NS2.produced, anime_uri))

    # === Staff (directores, productores, etc.) ===
    try:
        staff_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/staff").json()['data']
        for s in staff_data:
            staff_uri = ONTO[clean_uri(s['person']['name'])]
            if (staff_uri, RDF.type, NS2.RealPerson) not in g:
                g.add((staff_uri, RDF.type, NS2.RealPerson))
                g.add((staff_uri, NS2.label, Literal(s['person']['name'], datatype=XSD.string)))
            # Cada posición como rol de producción
            for pos in s.get('positions', []):
                role_uri = ONTO[clean_uri(pos)]
                if (role_uri, RDF.type, NS2.ProductionRole) not in g:
                    g.add((role_uri, RDF.type, NS2.ProductionRole))
                    g.add((role_uri, NS2.label, Literal(pos, datatype=XSD.string)))
                if (role_uri, NS2.roleIn, anime_uri) not in g:
                    g.add((role_uri, NS2.roleIn, anime_uri))
    except:
        pass

    # === Characters y VoiceActors ===
    try:
        char_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/characters").json()['data']
        for c in char_data:
            char_uri = ONTO[clean_uri(c['character']['name'])]
            if (char_uri, RDF.type, NS2.Character) not in g:
                g.add((char_uri, RDF.type, NS2.Character))
                g.add((char_uri, NS2.label, Literal(c['character']['name'], datatype=XSD.string)))
            if (c['role'].lower() == 'main'):
                if (char_uri, RDF.type, NS2.MainCharacter) not in g:
                    g.add((char_uri, RDF.type, NS2.MainCharacter))
            if (char_uri, NS2.characterIn, anime_uri) not in g:
                g.add((char_uri, NS2.characterIn, anime_uri))
            # Voice actors
            for va in c.get('voice_actors', []):
                if va['language'].lower() != 'japanese':
                    continue
                va_uri = ONTO[clean_uri(va['person']['name'])]
                if (va_uri, RDF.type, NS2.VoiceActor) not in g:
                    g.add((va_uri, RDF.type, NS2.VoiceActor))
                    g.add((va_uri, NS2.label, Literal(va['person']['name'], datatype=XSD.string)))
                if (char_uri, NS2.dubbedBy, va_uri) not in g:
                    g.add((char_uri, NS2.dubbedBy, va_uri))
    except:
        pass

    # === Episodios ===
    try:
        eps_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/episodes").json()['data']
        for ep in eps_data:
            ep_uri = ONTO[clean_uri(ep['title'])]
            if (ep_uri, RDF.type, NS2.Movie) not in g:
                g.add((ep_uri, RDF.type, NS2.Episode))  # o ns2:Episode según tu ontología
            if (ep_uri, NS2.episodeTitle, Literal(ep['title'], datatype=XSD.string)) not in g:
                g.add((ep_uri, NS2.episodeTitle, Literal(ep['title'], datatype=XSD.string)))
            if (ep_uri, NS2.isEpisodeOf, anime_uri) not in g:
                g.add((ep_uri, NS2.isEpisodeOf, anime_uri))
    except:
        pass

    print(f"Procesado: {title} (ID {anime_id})")
    time.sleep(0.5)

# Guardar grafo actualizado
g.serialize("grafo_actualizado_completo.ttl", format="turtle")
print("¡Grafo completamente enriquecido listo!")
sbc.show_graph(g, output_file="grafo_actualizado_completo.html")


Procesado: Cowboy Bebop (ID 1)
Anime ID 2 error
Anime ID 3 error
Anime ID 4 error
Anime ID 5 error
Procesado: Trigun (ID 6)
Procesado: Witch Hunter Robin (ID 7)
Procesado: Beet the Vandel Buster (ID 8)
